### Day20 - RAGAS Evaluation

In [5]:
# Uncomment if needed

# !pip install ragas
# !pip install datasets
# !pip install langchain-groq
# !pip install litellm

FILE_NAME = 'SAMPLE_RAG_Ground_Truth_Eval'

In [2]:
import json
import pandas as pd

from datasets import Dataset
from langchain_huggingface import HuggingFaceEmbeddings

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

from langchain_groq import ChatGroq

In [9]:
# INPUT_JSON = "data/Test_Questions_Eval.json"
INPUT_JSON = "./data/%s.json" % FILE_NAME

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} evaluation samples")

Loaded 39 evaluation samples


In [10]:
df = pd.DataFrame(data)
display(df)

,answer,context,question,expected_source,ground_truth_answer
0,The probation period for new full-time employe...,[Employees may be classified into the followin...,What is the probation period for new full-time...,employee_handbook,All new Full-Time employees undergo a standard...
1,I couldn't find this information in the policies.,[Economy class is standard. Business class req...,What are the standard office hours?,employee_handbook,"Monday to Friday, 9:00 AM–6:00 PM (IST)."
2,"According to the context, eligible employees m...","[Standard office hours are Monday to Friday, 9...",How many days per week can eligible employees ...,employee_handbook,Up to three days per week with manager approval.
3,VPN must be used whenever accessing corporate ...,[used whenever accessing corporate resources f...,Is VPN required for remote access?,employee_handbook,Yes. Company-approved corporate VPN is mandato...
4,"According to the context, full-time employees ...",[tems remotely.\n4 Time Off and Leave Policy\n...,How many annual leave days do full-time employ...,employee_handbook,18 days.
5,"According to the context, ""Medical Leave: 10 d...",[tems remotely.\n4 Time Off and Leave Policy\n...,How many medical leave days are provided?,employee_handbook,10 days.
6,I couldn't find this information in the policies.,[3.1 Working Hours . . . . . . . . . . . . . ....,How often are performance reviews conducted?,employee_handbook,"Twice yearly, in June and December."
7,I couldn't find this information in the policies.,[3.1 Working Hours . . . . . . . . . . . . . ....,What is the annual learning budget?,employee_handbook,"$1,500 (or local equivalent)."
8,I couldn't find this information in the policies.,[up to termination. Violations may result in c...,How often must passwords be rotated?,employee_handbook,Every 90 days.
9,I couldn't find this information in the policies.,[up to termination. Violations may result in c...,What is the resignation notice period?,employee_handbook,30 days for full-time employees.


In [11]:
ragas_rows = []

for row in data:
    # print("question" in row.keys())
    if "question" in row.keys():
        ragas_rows.append({
            "question": row["question"],
            "answer": row["answer"],
            "contexts": row["context"],
            "ground_truth": row["ground_truth_answer"]
        })

dataset = Dataset.from_list(ragas_rows)

display(dataset)

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 21
})

In [6]:
# import os
# from dotenv import load_dotenv
# # 1. Load the environment variables from your .env file
# loaded = load_dotenv()
# if not loaded:
#     raise ValueError("No keys found! Please check your .env file.")
#
# env = open('./.env', 'r')
# #print(env.read())
# print(os.getenv('GROQ_API_KEY2'))
#
# # Verify the key is loaded (don't print the actual key in output!)
# # if not os.getenv("GROQ_API_KEY"):
# #     raise ValueError("GROQ_API_KEY not found. Please check your .env file.")
#
# judge_llm = ChatGroq(
#     model="llama-3.1-70b-versatile",
#     temperature=0
# )

In [12]:
import os
from dotenv import load_dotenv

# 1. Load the environment variables from your .env file
load_dotenv(override=True)

# Verify the key is loaded (don't print the actual key in output!)
if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY not found. Please check your .env file.")

# 2. Initialize Models
# Requires GROQ_API_KEY in your .env file
judge_model = ChatGroq(
    # CHECK: https://console.groq.com/docs/deprecations
    # model="llama-3.1-8b-instant",
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=4096
)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print(f"✅ Judge LLM configured as: {judge_model.model_name}")
print(f"✅ Embeddings configured as: {embeddings.model_name}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Judge LLM configured as: llama-3.3-70b-versatile
✅ Embeddings configured as: sentence-transformers/all-MiniLM-L6-v2


In [13]:
# Execute Ragas using the cloud models
score = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    llm=judge_model,
    embeddings=embeddings,
    # Prevents the entire run from failing if one evaluation errors out
    raise_exceptions=False,
    # Prevents overload
    batch_size=4
)

# Output as a clean Pandas DataFrame for analysis
results_df = score.to_pandas()
display(pd.DataFrame(results_df))
print()

Evaluating:   0%|          | 0/84 [00:00<?, ?it/s]

Batch 1/21:   0%|          | 0/4 [00:00<?, ?it/s]

Exception raised in Job[42]: TimeoutError()
Exception raised in Job[47]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kvmqhvbte61vev1wapfyx37n` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99800, Requested 1247. Please try again in 15m4.608s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
Exception raised in Job[45]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kvmqhvbte61vev1wapfyx37n` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99764, Requested 539. Please try again in 4m21.792s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
Exception raised in Job[44]: RateLimitError(Error

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is the probation period for new full-time...,[Employees may be classified into the followin...,The probation period for new full-time employe...,All new Full-Time employees undergo a standard...,1.0,1.000000,1.0,1.0
1,What are the standard office hours?,[Economy class is standard. Business class req...,I couldn't find this information in the policies.,"Monday to Friday, 9:00 AM–6:00 PM (IST).",1.0,0.000000,0.0,0.0
2,How many days per week can eligible employees ...,"[Standard office hours are Monday to Friday, 9...","According to the context, eligible employees m...",Up to three days per week with manager approval.,1.0,0.884653,1.0,1.0
3,Is VPN required for remote access?,[used whenever accessing corporate resources f...,VPN must be used whenever accessing corporate ...,Yes. Company-approved corporate VPN is mandato...,1.0,0.855592,1.0,1.0
4,How many annual leave days do full-time employ...,[tems remotely.\n4 Time Off and Leave Policy\n...,"According to the context, full-time employees ...",18 days.,1.0,0.983338,1.0,1.0
5,How many medical leave days are provided?,[tems remotely.\n4 Time Off and Leave Policy\n...,"According to the context, ""Medical Leave: 10 d...",10 days.,1.0,0.931676,1.0,1.0
6,How often are performance reviews conducted?,[3.1 Working Hours . . . . . . . . . . . . . ....,I couldn't find this information in the policies.,"Twice yearly, in June and December.",1.0,0.000000,0.0,0.0
7,What is the annual learning budget?,[3.1 Working Hours . . . . . . . . . . . . . ....,I couldn't find this information in the policies.,"$1,500 (or local equivalent).",1.0,0.000000,0.0,0.0
8,How often must passwords be rotated?,[up to termination. Violations may result in c...,I couldn't find this information in the policies.,Every 90 days.,0.0,0.000000,0.0,0.0
9,What is the resignation notice period?,[up to termination. Violations may result in c...,I couldn't find this information in the policies.,30 days for full-time employees.,1.0,0.000000,0.0,0.0


### ALTERNATE
#### OpenAI in lieu of Groq/HF

In [9]:
import os
import pandas as pd
from dotenv import load_dotenv

# Ensure you have imported the OpenAI equivalents from LangChain
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

# 1. Load the environment variables from your .env file
load_dotenv(override=True)

# Verify the OpenAI key is loaded (don't print the actual key in output!)
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Please check your .env file.")

# 2. Initialize Models
# Requires OPENAI_API_KEY in your .env file

# Using gpt-4o-mini as a fast, cost-effective judge model
judge_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=4096
)

# Using OpenAI's native embeddings
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print(f"✅ Judge LLM configured as: {judge_model.model_name}")
print(f"✅ Embeddings configured as: {embeddings.model}")

✅ Judge LLM configured as: gpt-4o-mini
✅ Embeddings configured as: text-embedding-3-small


In [10]:
# 3. Execute Ragas using the OpenAI cloud models
# Assuming 'dataset' is defined prior to this snippet
score = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    llm=judge_model,
    embeddings=embeddings,
    # Prevents the entire run from failing if one evaluation errors out
    raise_exceptions=False,
    # Prevents overload
    batch_size=4
)

# Output as a clean Pandas DataFrame for analysis
results_df = score.to_pandas()
display(pd.DataFrame(results_df))
print()

Evaluating:   0%|          | 0/84 [00:00<?, ?it/s]

Batch 1/21:   0%|          | 0/4 [00:00<?, ?it/s]

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is the probation period for new full-time...,[Employees may be classified into the followin...,The probation period for new full-time employe...,All new Full-Time employees undergo a standard...,1.0,1.000000,1.0,1.0
1,What are the standard office hours?,[Economy class is standard. Business class req...,I couldn't find this information in the policies.,"Monday to Friday, 9:00 AM–6:00 PM (IST).",1.0,0.000000,0.0,0.0
2,How many days per week can eligible employees ...,"[Standard office hours are Monday to Friday, 9...","According to the context, eligible employees m...",Up to three days per week with manager approval.,1.0,1.000000,1.0,1.0
3,Is VPN required for remote access?,[used whenever accessing corporate resources f...,VPN must be used whenever accessing corporate ...,Yes. Company-approved corporate VPN is mandato...,0.5,0.840783,1.0,1.0
4,How many annual leave days do full-time employ...,[tems remotely.\n4 Time Off and Leave Policy\n...,"According to the context, full-time employees ...",18 days.,1.0,0.936426,1.0,1.0
5,How many medical leave days are provided?,[tems remotely.\n4 Time Off and Leave Policy\n...,"According to the context, ""Medical Leave: 10 d...",10 days.,1.0,0.738983,1.0,1.0
6,How often are performance reviews conducted?,[3.1 Working Hours . . . . . . . . . . . . . ....,I couldn't find this information in the policies.,"Twice yearly, in June and December.",0.0,0.000000,0.0,0.0
7,What is the annual learning budget?,[3.1 Working Hours . . . . . . . . . . . . . ....,I couldn't find this information in the policies.,"$1,500 (or local equivalent).",0.0,0.000000,0.0,0.0
8,How often must passwords be rotated?,[up to termination. Violations may result in c...,I couldn't find this information in the policies.,Every 90 days.,0.0,0.000000,0.0,0.0
9,What is the resignation notice period?,[up to termination. Violations may result in c...,I couldn't find this information in the policies.,30 days for full-time employees.,0.0,0.000000,0.0,0.0


In [17]:
import pandas as pd

# 1. Create a copy of the dataframe so we don't modify the original one in your notebook
export_df = results_df.copy()

# Note: Ragas typically names this column 'contexts', but we'll check for 'context' as well
target_col = 'contexts' if 'contexts' in export_df.columns else 'context'

# 2. Convert the list of strings into a single string separated by double newlines for readability
if target_col in export_df.columns:
    export_df[target_col] = export_df[target_col].apply(
        lambda x: "\n\n---\n\n".join(x) if isinstance(x, list) else x
    )

# 3. Save to Excel
excel_filename = "data/%s.xlsx" % FILE_NAME

# We use the openpyxl engine and set index=False so we don't get the arbitrary row numbers in the output
export_df.to_excel(excel_filename, index=False, engine='openpyxl')

print(f"✅ Results successfully formatted and saved to {excel_filename}")

✅ Results successfully formatted and saved to data/RAG_Ground_Truth_Eval.xlsx
